In [2]:
import aiohttp
import ast
import asyncio
import openai
import pandas as pd
import re
import json

from openai import AsyncClient
from datasets import load_dataset
from tqdm.asyncio import tqdm

In [ ]:
async_client = AsyncClient()

In [4]:
dataset = load_dataset("DataForGood/climateguard")
df = pd.concat([dataset["train"].to_pandas(), dataset["test"].to_pandas()])
df.start = pd.to_datetime(df.start).dt.tz_convert('Europe/Paris')

In [5]:
df.start.min(), df.start.max()

(Timestamp('2023-05-01 16:26:00+0200', tz='Europe/Paris'),
 Timestamp('2025-11-19 14:42:00+0100', tz='Europe/Paris'))

In [6]:
df_train = df.loc[
    (df.start >= pd.to_datetime("2025-01-01").tz_localize('Europe/Paris')) & \
    (df.start < pd.to_datetime("2025-07-01").tz_localize('Europe/Paris')) & \
    (df.claims.map(len) > 0)
]
df_test = df.loc[(df.start >= pd.to_datetime("2025-07-01").tz_localize('Europe/Paris'))]

In [7]:
display(df_train.claims.map(len).value_counts())
display(df_test.claims.map(len).value_counts())

claims
1    213
2     83
3     31
4      7
5      4
6      1
Name: count, dtype: int64

claims
0    470
1    129
2     53
3     19
4      2
Name: count, dtype: int64

In [8]:
len(".\n".join(df_train.claims.map(lambda x: ".\n".join(x)).tolist()))

56139

In [9]:
claims = pd.read_csv("id2clusters.csv")
def get_unique_clusters(claims):
    clusters = claims.Clusters.unique().tolist()
    unique_clusters = set()
    for cluster in clusters:
        try:
            if isinstance(cluster, str):
                sub_cluster = ast.literal_eval(cluster)
            elif isinstance(cluster, list):
                sub_cluster = cluster
            else:
                raise TypeError
            for _cluster in sub_cluster:
                unique_clusters.add(_cluster)
        except TypeError as e:
            print(cluster)
            print(type(cluster))
            raise e
    return unique_clusters

def get_size_unique_clusters_from_series(clusters):
    unique_clusters = set()
    for cluster in clusters:
        try:
            if isinstance(cluster, str):
                sub_cluster = ast.literal_eval(cluster)
            elif isinstance(cluster, list):
                sub_cluster = cluster
            else:
                raise TypeError
            for _cluster in sub_cluster:
                unique_clusters.add(_cluster)
        except TypeError as e:
            print(cluster)
            print(type(cluster))
            raise e
    return len(unique_clusters)
    

In [10]:
import plotly.express as px

for group in claims.groupby("Month").agg({"Clusters": get_size_unique_clusters_from_series}):
    print(group)
    break

Clusters


In [11]:
px.bar(claims.groupby("Month").agg({"Clusters": get_size_unique_clusters_from_series}))

In [12]:
claims_train = claims.loc[pd.to_datetime(claims.Start) < pd.to_datetime("2025-06-01").tz_localize('Europe/Paris')]
unique_clusters = get_unique_clusters(claims_train)
len(unique_clusters)

/var/folders/_y/jjyv0bm5597_x15dvpxnsmrc0000gn/T/ipykernel_7161/121838707.py:1: FutureWarning:

In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`



18

5.0

In [21]:
df_frugal_ai_claims = pd.read_csv("frugalaiclaims.csv")
cutoff = df_frugal_ai_claims.cluster.value_counts().describe()["75%"].item()
print(cutoff)
additional_claims = df_frugal_ai_claims.cluster.value_counts()[df_frugal_ai_claims.cluster.value_counts() > cutoff].index.tolist()
additional_claims = set(additional_claims)
len(additional_claims)

5.0


26

In [23]:
unique_clusters = unique_clusters.union(additional_claims)
len(unique_clusters)

44

In [24]:
async def get_response(
        user_prompt, 
        system_prompt=None, 
        model="gpt-5-nano", 
        reasoning_effort="minimal", 
        temperature=0.1, 
        semaphore=None,
        max_output_tokens=512,
        structured_output=False,
        idx=None,
    ):
    messages = []
    if system_prompt:
        messages.append(
            {
                "role": "system", 
                "content": system_prompt,
            }
        )
    messages.append(
        {
            "role": "user", 
            "content": user_prompt,
        }
    )
    if semaphore:
        await semaphore.acquire()
    try:
        response_args = dict(
            model=model,
            input=messages,
            max_output_tokens=max_output_tokens,
        )
        if '5' in model:
            response_args.update(dict(reasoning={"effort": reasoning_effort}))
        else:
            response_args.update(dict(temperature=temperature))
        chat_response = await async_client.responses.create(
            **response_args
        )
    finally:
        if semaphore:
            semaphore.release()
    try:
        if structured_output:
            return idx, ast.literal_eval(chat_response.output_text)
        else:
            return idx, chat_response.output_text
    except SyntaxError:
        return idx, str(chat_response.output_text)

In [25]:
system_prompt = "You are an assistant helping editors to moderate TV and radio content and detect misinformation."
# If a claim is contained but contraddicted by another voice, respond NO.
prompt = """
Given a transcript from a TV or radio show, you are to perform a claim matching task.
We want to identify texts promoting climate change misinformation that undermines well-established scientific consensus, 
such as denying the existence of climate change or the factors that contribute to it.

This transcript is very messy and contains no punctuation, as it comes from a basic speech to text model.
You are given a list of claims that have been extracted from disinformation narratives over the past months, 
if such a claim is contained in the transcript, if the content of the transcription is very similar to one of
the claims, or if the intent of the speaker is to validate the one of the claims return YES. 
If you identify the transcript as containing a case of misinformation then return YES.
Otherwise respond NO. 

The the misinformation claims are often not contained directly, but are implied by the speaker.
Respond only YES or NO.
Here is the list of claims: 
{claims}
And here is the transcript:
{transcript}
"""
def map_classification(classification):
    if classification.lower() == "yes":
        return True
    elif classification.lower() == "no":
        return False
    elif "no" in classification.lower():
        return False
    elif "yes" in classification.lower():
        return True
    return False


In [26]:
async def rewrite_and_classify(
    claims,
    transcript,
    rewrite_prompt,
    classify_prompt,
    system_prompt=None, 
    model="gpt-5-nano", 
    reasoning_effort="minimal", 
    temperature=0.1, 
    semaphore=None,
    structured_output=False,
    idx=None,
):
    idx, rewritten_transcript = await get_response(
        rewrite_prompt.format(transcript=transcript), 
        system_prompt=None, 
        model=model, 
        reasoning_effort=reasoning_effort, 
        temperature=temperature, 
        semaphore=semaphore,
        max_output_tokens=512,
        structured_output=False,
        idx=idx,
    )
    idx, response = await get_response(
        classify_prompt.format(claims=claims, transcript=rewritten_transcript), 
        system_prompt=system_prompt, 
        model=model, 
        reasoning_effort=reasoning_effort, 
        temperature=temperature, 
        semaphore=semaphore,
        max_output_tokens=16,
        structured_output=structured_output,
        idx=idx,
    )
    
    return idx, response

In [27]:
rewrite_prompt = """
You are an editor cleaning transcripts for downstream tasks.
You are given a transcript generated using a basic speech-to-text model. This is a messy transcript with no punctuation and no
diarization. You are to clean the transcript, add proper punctuation and infer the diarization. Do not change single words, 
do not remove stutters and expressions like 'uhm', 'bon' and 'eh'. Stay as loyal as possible to the original speech to text.
Do not add markdown to the transcript.
Here is the transcript:
{transcript}
"""

In [28]:
from sklearn.metrics import classification_report
from tqdm.asyncio import tqdm

import asyncio


async def test_prompt(
    df,
    user_prompt, 
    system_prompt=None, 
    model="gpt-5-nano", 
    reasoning_effort="minimal", 
    temperature=0.1, 
    semaphore=1,
    structured_output=False,
):
    semaphore = asyncio.Semaphore(semaphore)
    tasks = [
        get_response(
            user_prompt.format(claims=unique_clusters, transcript=row["plaintext"]), 
            system_prompt=system_prompt, 
            model=model, 
            reasoning_effort=reasoning_effort, 
            temperature=temperature, 
            semaphore=semaphore,
            structured_output=structured_output,
            idx=idx,
        ) for idx, row in df.iterrows()
    ]
    predictions = await tqdm.gather(*tasks)
    predictions = sorted(predictions, key=lambda x: x[0])
    print(predictions)
    df["predictions"] = [map_classification(element[1]) for element in predictions]
    return classification_report(
        df.misinformation.values,
        df.predictions.values
    )

In [31]:
report = await test_prompt(
    df_test.sample(n=100),
    prompt, 
    system_prompt=system_prompt, 
    model="gpt-4o-mini", 
    reasoning_effort="minimal", 
    temperature=0.7, 
    semaphore=50,
    structured_output=False,
)
print(report)

100%|██████████| 100/100 [00:05<00:00, 16.95it/s]

[(185, 'YES'), (187, 'NO'), (189, 'YES'), (191, 'NO'), (194, 'YES'), (202, 'YES'), (203, 'YES'), (234, 'YES'), (236, 'YES'), (240, 'NO'), (247, 'NO'), (249, 'YES'), (254, 'YES'), (427, 'NO'), (429, 'NO'), (430, 'YES'), (433, 'NO'), (442, 'YES'), (443, 'NO'), (445, 'NO'), (447, 'YES'), (448, 'YES'), (465, 'YES'), (481, 'NO'), (485, 'NO'), (488, 'NO'), (489, 'NO'), (490, 'NO'), (517, 'NO'), (520, 'YES'), (522, 'YES'), (529, 'YES'), (531, 'YES'), (533, 'YES'), (538, 'YES'), (540, 'NO'), (543, 'YES'), (575, 'YES'), (580, 'YES'), (582, 'NO'), (586, 'YES'), (604, 'YES'), (614, 'NO'), (638, 'YES'), (641, 'YES'), (649, 'YES'), (650, 'YES'), (657, 'NO'), (660, 'NO'), (662, 'NO'), (663, 'NO'), (670, 'NO'), (673, 'NO'), (685, 'NO'), (690, 'YES'), (695, 'YES'), (702, 'NO'), (730, 'YES'), (734, 'NO'), (735, 'NO'), (740, 'NO'), (1243, 'NO'), (1248, 'NO'), (1255, 'NO'), (1256, 'YES'), (1263, 'YES'), (1268, 'YES'), (1274, 'YES'), (1275, 'NO'), (1277, 'NO'), (1283, 'NO'), (1285, 'NO'), (1325, 'YES'), (